In [ ]:
# Install dependencies and add source directory to path

import sys, os

%pip install -r ./requirements.txt --quiet
sys.path.append('src')

In [ ]:
# Import libraries

import glob
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pylab as plt
from matplotlib.gridspec import GridSpec
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
import albumentations
import cv2

In [ ]:
# Import local utilities

from model_scss_net import scss_net
from data_loader import load_images, load_masks
from predictor import predict_batch
from metrics import compute_metrics, dice_np, iou_np, dice, iou
from pdf_export import create_pdf
from plot_utils import plot_imgs, plot_metrics
from augmentation import DataGenerator, create_augmentations

In [ ]:
# Load and count training images and annotations

imgs = sorted(glob.glob("./data/EL_6300_EPB_2023-10--12_revised/imgs/*.png"))
masks = sorted(glob.glob("./data/EL_6300_EPB_2023-10--12_revised/masks/*.png"))

print(f"Imgs number = {len(imgs)}\nMasks number = {len(masks)}")

In [ ]:
# Verify that each image is paired with the correct mask

mismatches = []
for i, (img_path, mask_path) in enumerate(zip(imgs, masks)):
    img_name = os.path.basename(img_path)
    mask_name = os.path.basename(mask_path)
    if img_name != mask_name:
        mismatches.append((i, img_name, mask_name))

if mismatches:
    print(f"WARNING: Found {len(mismatches)} mismatched pairs:")
    for idx, img, msk in mismatches[:10]:  # Show first 10
        print(f"  Index {idx}: '{img}' != '{msk}'")
else:
    print("All filenames match!")
    print(f"First pair: {os.path.basename(imgs[0])}")
    print(f"Last pair: {os.path.basename(imgs[-1])}")

In [ ]:
IMG_SIZE = 512  # Resize imgs to 512x512
BATCH_SIZE = 1  # Set batch size
SEED = 42       # Set seed for reproducibility
EPOCHS = 100    # Set number of epochs (can be increased when using EarlyStopping)

MODEL_NAME = "model_epb_final"                       # Specify model name
model_filename = f"./{MODEL_NAME}.h5"                # Specify path where to save model

In [ ]:
# Load images and masks, remove alpha channel from images, resize to IMG_SIZE x IMG_SIZE

imgs_list = []  # list of input images
masks_list = [] # list of annotations

for image, mask in zip(imgs, masks):
    # Alpha channel removal
    rgba = Image.open(image).convert("RGBA")
    background = Image.new("RGBA", rgba.size, (0, 0, 0, 255))
    composited = Image.alpha_composite(background, rgba)

    # Convert to grayscale, resize and append to lists
    imgs_list.append(np.array(composited.convert("L").resize((IMG_SIZE, IMG_SIZE))))
    masks_list.append(np.array(Image.open(mask).convert("L").resize((IMG_SIZE, IMG_SIZE))))

In [ ]:
# Data normalization: from (0; 255) to (0; 1)
x = np.asarray(imgs_list, dtype=np.float32)/255
y = np.asarray(masks_list, dtype=np.float32)/255

# Reshape to (n_imgs, height, width, channels)
x = x.reshape(x.shape[0], x.shape[1], x.shape[2], 1)
y = y.reshape(y.shape[0], y.shape[1], y.shape[2], 1)

# Split data to Train set (90%) and Validation set (10%)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.1, random_state=None, shuffle=True)

In [ ]:
# Define geometric and color augmentations for training
# Rotation is disabled for EPB segmentation because EPB structures appear exclusively in a top-to-bottom orientation.

geometric_aug, color_aug = create_augmentations(
    horizontal_flip=True,
    vertical_flip=True, 
    rotation_limit=0,
    border_mode=cv2.BORDER_REPLICATE,
    separate_color=True
)

# Initialize training data generator
train_gen = DataGenerator(
    x_data=x_train,
    y_data=y_train,
    batch_size=BATCH_SIZE,
    augmentations=geometric_aug, 
    color_augmentations=color_aug,  
    shuffle=True,
    seed=SEED
)

# Preview a augmented sample
x_sample, y_sample = next(train_gen.generate())
plot_imgs(imgs=x_sample, masks=y_sample, n_imgs=2).show()

In [ ]:
# Input shape should be (512, 512, 1)
input_shape = x_train[0].shape
print(f"Input shape: {input_shape}\nTrain shape: {x_train.shape}  Val shape: {x_val.shape}")

In [ ]:
# Load model architecture with optimal parameteres
model = scss_net( 
    input_shape,
    filters=32,       
    layers=4,
    batch_norm=True,
    drop_prob=0.5)

# Compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",  
    metrics=[iou, dice])

# Set steps parameters acording to size of training set and size of batch
STEPS = x_train.shape[0] // BATCH_SIZE        

# Set Callback that saves only best weights
callback_checkpoint = ModelCheckpoint(
    model_filename,
    verbose=1,
    monitor="val_loss",
    save_best_only=True)


# To train with EarlyStopping, uncomment the block below

# from tensorflow.keras.callbacks import EarlyStopping
# callback_earlystop = EarlyStopping(
#     monitor='val_loss',         # Monitor validation loss
#     patience=30,                # Stop training after 30 epochs without improvement
#     restore_best_weights=True,  # Restore best weights after training
#     verbose=1
# )

In [ ]:
# Train model
# To use EarlyStopping, replace callbacks with: [callback_checkpoint, callback_earlystop]
history = model.fit(
    train_gen.generate(),
    steps_per_epoch=STEPS,
    epochs=EPOCHS,
    validation_data=(x_val, y_val),
    callbacks=[callback_checkpoint],
    verbose=2)

In [ ]:
# Plot training history (Metrics and Loss)
plot_metrics(history).show()

## Results

In [ ]:
# Load the trained model
model.load_weights(model_filename)

### Results on validation set

In [ ]:
# Predict new masks for validation set
y_pred = model.predict(x_val)

In [ ]:
# Vizualization of results
plot_imgs(imgs=x_val, masks=y_val, predictions=y_pred, n_imgs=2).show()

In [ ]:
# Calculate averaged metrics (Dice and IoU) over the validation set without and with applied threshold
compute_metrics(y_val, y_pred, threshold=0.5)

### Results on test set / Deployment

**The following section can be used to display results on the test set or display
deployment results, i.e. predictions on data that do not have ground-truth masks.**

- If ground-truth masks are available, set MASKS_DIR to the directory containing them.\
The script will then compute Dice and IoU metrics and generate a 4-column PDF (image | ground-truth mask | predicted mask | overlay).

- If no masks are available (deployment), set MASKS_DIR = "" \
The script will skip metric computation and generate a 3-column PDF (image | predicted mask | overlay).

In [ ]:
# Input and output paths

IMG_DIR = "./data/test/imgs"
MASKS_DIR = "./data/test/masks"
PREDICTIONS_DIR = "./data/test/predictions"
PDF_PATH = "predictions_2.pdf"

In [ ]:
# Load data

imgs_test_list, img_names, x_test = load_images(IMG_DIR, img_size=IMG_SIZE)
masks_list, y_test = load_masks(MASKS_DIR, img_names, img_size=IMG_SIZE)

In [ ]:
# Batch prediction

y_pred = predict_batch(
    model=model,
    x_test=x_test,
    img_names=img_names,
    predictions_dir=PREDICTIONS_DIR,
    batch_size=BATCH_SIZE,
)

In [ ]:
# Calculate averaged metrics over the set without and with applied threshold
# No-op if masks are not available (y_test is None)
compute_metrics(y_test, y_pred, threshold=0.5)

In [ ]:
# Vizualization of results
plot_imgs(imgs=x_test, masks=y_test, predictions=y_pred, n_imgs=5).show()

In [ ]:
# Generate pdf:
# if ground-truth masks are available -> generated pdf will have 4 columns (original, mask, prediction, overlay)
# if no ground-truth masks are available -> generated pdf will have 3 columns (original, prediction, overlay)

create_pdf(
    pdf_path=PDF_PATH,
    imgs_test_list=imgs_test_list,
    img_names=img_names,
    x_test=x_test,
    y_pred=y_pred,
    masks_list=masks_list,   # None - 3 columns, otherwise - 4 columns
    rows_per_page=4,
)
